In [1]:
import pandas as pd
from google.cloud import bigquery
import logging

def etl_pipeline(request):
    try:
        # Caminho (ajuste depois se quiser usar bucket)
        file_path = '/tmp/sales_raw.csv'

        # Simulação: você pode evoluir para Cloud Storage depois
        df = pd.read_csv(file_path)

        # Limpeza
        df = df.dropna().drop_duplicates()
        df['Data Pedido'] = pd.to_datetime(df['Data Pedido'])

        # Dimensões
        dim_customer = df[['ID Cliente', 'Nome Cliente']].drop_duplicates()
        dim_product = df[['ID Produto', 'Nome Produto']].drop_duplicates()

        # Fato
        fact_sales = df[['Data Pedido', 'ID Cliente', 'ID Produto', 'Vendas']]

        # BigQuery
        client = bigquery.Client()
        dataset = "gleaming-block-492522-v5.dw_vendas"

        client.load_table_from_dataframe(dim_customer, f"{dataset}.dim_customer").result()
        client.load_table_from_dataframe(dim_product, f"{dataset}.dim_product").result()
        client.load_table_from_dataframe(fact_sales, f"{dataset}.fact_sales").result()

        return "Pipeline executado com sucesso!"

    except Exception as e:
        logging.error(f"Erro no pipeline: {e}")
        return f"Erro: {e}"